<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Obstacle Avoidance


Finally, we will use our object detection model to avoid duckie pedestrians as we navigate in Duckietown. 

The file that we will use for this [`model.py`](../../packages/solution/model.py).

The function that we need to write, with some helper functions, is `get_wheel_velocities_from_image`. You can expect that this will get called every time a new camera image comes in, and we should return wheel commands. The strategy we will take is very simple, if we detect a duckie pedestrian with sufficiently high confidence and sufficiently close, then we will stop (send wheel commands of 0), otherwise we will send fixed wheel command values to make the robot go (approximately in the real Duckiebot case) straight. 


The structure of this function is as follows:

```python
    def get_wheel_velocities_from_image(self, img: np.ndarray):
        try:
            detections = self._run_detector(img)
        except Exception as e:
            print(f"Inference error: {e}")
            return [DifferentialPWM(left=0.0, right=0.0), None]
        if self._should_stop(detections):
            return [DifferentialPWM(left=0.0, right=0.0), detections]
        else:
            return [DifferentialPWM(left=FORWARD_PWM, right=FORWARD_PWM), detections]
```

The function returns the wheel commands together with the detections, so that the agent can draw them on the debug image (see the end of this notebook).
Note that `FORWARD_PWM` is defined in your [solution/config.py](../../packages/solution/config.py). This will determine how fast your robot moves when there are (hopefully) no duckies in the way.

There are two function calls we make, the first is `_run_detector`.
It preprocesses the image, runs it through the model, and then calls `_postprocess` to turn the raw predictions into clean detections:

```python
    def _run_detector(self, img_bgr):
        x = self._preprocess(img_bgr)
        raw = self.backend.infer(x)
        return self._postprocess(raw)
```


### Post-processing: why we need NMS

YOLO works by dividing the image into a grid and making one or more bounding-box predictions at every grid cell - called **anchors**. For a 480×640 image with the nano model, this produces roughly **8 000 candidate boxes**, most of which overlap heavily.

We want **one** clean box per real object, not thousands of duplicates. **Non-Maximum Suppression (NMS)** does exactly that:

1. Discard all candidates whose confidence score is below a threshold (`CONF_THRESHOLD`).
2. From the remaining boxes, keep the one with the highest score.
3. Suppress every other box that overlaps the kept box by more than an IoU (Intersection-over-Union) threshold.
4. Repeat from step 2 for the next highest-scoring surviving box.

**IoU** measures how much two boxes overlap:

```
        area of overlap
IoU = ─────────────────────
        area of union
```

IoU = 0 means no overlap at all, IoU = 1 means the boxes are identical.


#### Raw YOLO output format

When the model is exported **without** built-in NMS (`nms=False`, which is how the training notebooks export it), the raw tensor has shape `[4 + C, num_anchors]` where `C` is the number of classes. Each column is one anchor:

| Column index | Meaning |
|---|---|
| 0 | center x (`cx`) in pixels |
| 1 | center y (`cy`) in pixels |
| 2 | box width (`w`) in pixels |
| 3 | box height (`h`) in pixels |
| 4 … 4+C-1 | confidence score for each class |

Your job is to implement `_apply_nms`, which turns this raw tensor into a clean `[M, 6]` array of detections - one row per surviving box:

| Column | Meaning |
|---|---|
| 0–3 | `x1, y1, x2, y2` (corner format) |
| 4 | confidence score |
| 5 | class id |



```python
    def _apply_nms(self, raw: np.ndarray, iou_thr: float = 0.45) -> np.ndarray:
        """
        Apply Non-Maximum Suppression to the raw YOLO anchor output.
        raw: [4+C, anchors] - once transposed to [anchors, 4+C] the columns are: cx, cy, w, h, score_class0, ...
             (pixel coordinates in input image space)
        Returns [M, 6]  (x1, y1, x2, y2, score, class_id).
        """

        # TODO 1. Transpose raw to [anchors, 4+C]
        #         Extract cx, cy, w, h (columns 0-3) and class scores (columns 4+).
        #         Hint: class_scores.argmax(axis=1) gives the most-likely class per anchor,
        #               index into the scores array to get the winning score.
        #      2. Filter anchors below CONF_THRESHOLD
        #         If nothing passes the threshold, return np.zeros((0, 6), np.float32).
        #      3. Convert center format to corner format:
        #         cx/cy/w/h to x1/y1/x2/y2
        #      4. Run cv2.dnn.NMSBoxes to suppress overlapping boxes
        #         It expects boxes as a list of [x1, y1, w, h] (NOT x2/y2).
        #         If no boxes survive, return np.zeros((0, 6), np.float32).
        #      5. Assemble the surviving detections into an [M, 6] array:
        #         columns: x1, y1, x1+w, y1+h, score, class_id

        return np.zeros((0, 6), np.float32)  # placeholder: replace it with your implementation
```

***Tip:*** `cv2.dnn.NMSBoxes(boxes, scores, score_threshold, nms_threshold)` returns the *indices* of the kept boxes. Call `.flatten()` on the result before using it as an index array.


Your job will be to process the detections and decide whether we should stop or not. For each detection if the following two conditions are met you should stop:
1. the confidence (score) is higher than the `CONF_THRESHOLD` defined in [solution/config.py](../../packages/solution/config.py).
2. the duckie is closer than `STOP_DISTANCE` away. You should figure out a way to estimate how far away it is. 

The code looks like this:

```python
    def _should_stop(self, detections: np.ndarray): 
        
        stop = False

        for x1, y1, x2, y2, score, _ in detections:


            # TODO we don't want to consider detections with confidence (score) below CONF_THRESHOLD (a value you should set in config.py)
            

            # TODO we want to stop if there is a duckie closer than STOP_DISTANCE away
            # To calculate if the duckie is too close we need to convert the pixel coordinates to 
            # world coordinates. To do so you can use the `self.ground_projector` object which has
            # loaded the camera extrinsic calibration
            # Specifically, if you want to project an object of type `pix = Pixel(x=u, y=v)` to a ground plane
            # point, you can first convert it to a vector (`vec = self.ground_projector.camera.pixel2vector(pix)`) and
            # then you can intersect that vector with the ground plane (`self.ground_projector.vector2ground(vec)`). 
            # That will be the point on the ground plane corresponding to the input pixel. 

        return stop
```

Once you are happy with your implementation you can test it. As described in the [`README`](../../README.md) you should first build

```
dts code build -R ROBOT_NAME [--local]
```

And then run with 

```
dts code workbench [-m] -R ROBOT_NAME [--local] 
```

where you should include `-m` if you are running a virtual robot in the Duckiematrix.

If `ROBOT_NAME` is a **real** Duckiebot, the object detector runs on the robot's CPU by default. This works, but it is slow (a few frames per second on a Jetson Nano). Add the `--local` flag to both commands to build and run the code on your laptop instead, which is faster and is the recommended option. The flag is not needed for a virtual robot.

It can be useful to visualize the detections that are being made by your model. You can do so with the Image Viewer. Open it with 

```bash
dts duckiebot image_viewer ROBOT_NAME
```

In the top drop down you should be able to select a topic name `/ROBOT_NAME/object_detector_image/jpeg`. If you do so, that will show you an image with the bounding boxes overlayed as well as their associated confidences:

![image viewer object detection](../../assets/images/duckie_detection.png)


If all goes well your Duckiebot should STOP before any duckies get injured! 
Have fun exploring Duckietown safely!